# Therapeutic areas

Each disease ontology term is assigned one therapeutic area: the first therapeutic-area
root among its ontology ancestors, following the hierarchy in
`manuscript_methods.paper.THERAPEUTIC_AREAS`. Terms descending from no root become
`other`. Methods "Therapeutic area assignment to studies".

Writes `efo_therapeutic_area` (term to area) and `study_therapeutic_areas` (every GWAS
study with its areas, one-hot columns and the measurement flag).

In [ ]:
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from pyspark.sql import functions as f

from manuscript_methods import paper

session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

In [ ]:
studies = session.spark.read.parquet(paper.release("study"))
disease = session.spark.read.parquet(paper.release("disease") + "/disease.parquet").select("id", "ancestors")

# The hierarchy is a priority order, so coalesce picks the first area present in the ancestors.
first_area = f.coalesce(
    *[f.when(f.array_contains("ancestors", area), f.lit(area)) for area in paper.THERAPEUTIC_AREAS],
    f.lit("other"),
)

efo_ta = (
    disease.withColumn("primaryTherapeuticArea", first_area)
    .join(studies.select(f.explode("diseaseIds").alias("efo")), f.col("id") == f.col("efo"), "semi")
    .select("id", "primaryTherapeuticArea")
)
efo_ta.write.mode("overwrite").parquet(paper.derived("efo_therapeutic_area"))
print("ontology terms used by studies:", efo_ta.count())

## GWAS studies annotated with their therapeutic areas

In [ ]:
efo_ta = session.spark.read.parquet(paper.derived("efo_therapeutic_area"))

study_areas = (
    studies.select("studyId", f.explode("diseaseIds").alias("id"))
    .join(efo_ta, on="id", how="inner")
    .groupBy("studyId")
    .agg(f.collect_set("primaryTherapeuticArea").alias("mappedTherapeuticAreas"))
)

gwas = (
    studies.filter(f.col("studyType") == "gwas")
    .join(study_areas, on="studyId", how="left")
    .withColumn("measurement", f.array_contains("mappedTherapeuticAreas", paper.MEASUREMENT))
    .withColumn("binaryLessCases", f.col("nCases") < f.col("nControls"))
    .withColumns(
        {
            column: f.when(f.array_contains("mappedTherapeuticAreas", area), 1).otherwise(0)
            for area, column in paper.TA_COLUMNS.items()
        }
    )
)
gwas = gwas.withColumn("totalTherapeuticAreas", sum(f.col(c) for c in paper.TA_COLUMNS.values()))
gwas.write.mode("overwrite").parquet(paper.derived("study_therapeutic_areas"))
print("GWAS studies:", session.spark.read.parquet(paper.derived("study_therapeutic_areas")).count())

## Cross-check against the pre-refactor table

In [ ]:
new = session.spark.read.parquet(paper.derived("study_therapeutic_areas"))
ref = session.spark.read.parquet(paper.baseline("gwas_w_therapeutic_areas"))
columns = ["studyId", "measurement", "binaryLessCases", "totalTherapeuticAreas", *paper.TA_COLUMNS.values()]
print("rows:", new.count(), ref.count())
print("rows that differ:", new.select(columns).subtract(ref.select(columns)).count())